In [5]:
from Bio import SeqIO
import pandas as pd

mapping_df = pd.read_csv("../../documentation/clean_entity_mapping.csv")

# Set mit (pdb_id, chain_id) aus deinem Mapping für Hchain erstellen
heavy_chains = set()
for _, row in mapping_df.iterrows():
    pdb = str(row['pdb']).lower()
    h_chain = str(row['Hchain']).upper()
    if h_chain not in ['nan', '', 'None']:
        heavy_chains.add((pdb, h_chain))

print(f"Gefundene Heavy Chains im Mapping: {len(heavy_chains)}")

fasta_path = "../../documentation/pdb_sequences.fasta"
output_path = "hchain_sequences_deduplicated.fasta"

count_total = 0
unique_sequences = {}  # Sequenz → Header

def extract_pdb_and_chains(header):
    parts = header.split("_")
    pdb_id = parts[0].lower()
    chain_part = next((p for p in parts if p.startswith("chain")), None)
    if chain_part:
        chain_str = chain_part[len("chain"):].split("_")[0]
        chains = [c.upper() for c in chain_str.split(",")]
        return pdb_id, chains
    return pdb_id, []

for record in SeqIO.parse(fasta_path, "fasta"):
    header = record.id
    sequence = str(record.seq)
    count_total += 1

    pdb_id, chain_ids = extract_pdb_and_chains(header)
    if not chain_ids:
        continue

    # Prüfen, ob mindestens eine Chain in heavy_chains ist
    if any((pdb_id, chain) in heavy_chains for chain in chain_ids):
        if sequence not in unique_sequences:
            unique_sequences[sequence] = header  # ersten Header merken

# Deduplicated FASTA schreiben
with open(output_path, "w") as out_fasta:
    for seq, header in unique_sequences.items():
        out_fasta.write(f">{header}\n{seq}\n")

print(f"Insgesamt {count_total} Sequenzen geprüft.")
print(f"{len(unique_sequences)} eindeutige Heavy Chain Sequenzen in '{output_path}'")

Gefundene Heavy Chains im Mapping: 2596
Insgesamt 4318 Sequenzen geprüft.
1057 eindeutige Heavy Chain Sequenzen in 'hchain_sequences_deduplicated.fasta'


hier habe ich eine fasta Datei erstellt, die nur die Sequenzen der heavy chains beinhaltet. sie löscht außerdem alle doppelt vorkommenden Sequenzen, aber da bin ich nicht sicher, ob das notwendig ist. Außerdem wird hier auch nicht die target antigen species der jeweiligen hcain beachtet und auch die Paarung mit lchain nicht berücksichtigt.